# QQQI / QQQ / TQQQ v4.2 — State-2 Tail and Execution Diagnostics

This notebook evaluates the frozen v4.2 research baseline. It does **not** change signals, weights, thresholds, or the official 10 bps transaction-cost assumption.

Questions:

1. Are leveraged-state tail losses dominated by intraday deterioration or close-to-next-open gaps?
2. Were warning conditions observable before the worst losses?
3. How sensitive is v4.2 to one-session execution delays and additional slippage stress?
4. Does the pre-registered gate permit a continuous state-2 volatility-budget challenger?


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("artifacts/evidence/qqqi_qqq_tqqq_v4_2_state2_tail_diagnostics")
summary = json.loads((ROOT / "state_2_tail_summary.json").read_text(encoding="utf-8"))
episodes = pd.read_csv(ROOT / "state_2_episodes.csv", parse_dates=["start_date", "end_date", "trough_date", "worst_date"])
episode_summary = pd.read_csv(ROOT / "state_2_episode_summary.csv")
tail_days = pd.read_csv(ROOT / "state_2_top_tail_days.csv", parse_dates=["date"])
execution = pd.read_csv(ROOT / "execution_robustness.csv", index_col=0)

summary["economic_sample"], summary["research_gate"]

## 1. State-2 episode summary

In [ ]:
episode_summary.T

In [ ]:
episodes[[
    "episode_id", "start_date", "end_date", "sessions", "net_return",
    "max_drawdown", "overnight_loss_share", "worst_day_share_of_mae",
    "prior_close_warning_before_worst_day",
    "same_close_exit_signal_on_worst_day", "tail_mechanism",
]].sort_values("net_return")

In [ ]:
plot_data = episodes.sort_values("start_date")
plt.figure(figsize=(11, 5))
plt.bar(plot_data["episode_id"].astype(str), plot_data["net_return"] * 100)
plt.axhline(0, linewidth=1)
plt.xlabel("State-2 episode")
plt.ylabel("Net return (%)")
plt.title("State-2 episode net returns")
plt.tight_layout()
plt.show()

## 2. Worst state-2 sessions and loss decomposition

In [ ]:
tail_days[[
    "date", "net_return", "intraday_contribution", "overnight_contribution",
    "previous_close_warning", "same_close_exit_signal", "decision_reason",
    "vix_close", "vxn_close", "qqq_below_ma20",
]]

In [ ]:
components = tail_days[["intraday_contribution", "overnight_contribution"]].sum() * 100
plt.figure(figsize=(7, 4))
plt.bar(components.index, components.values)
plt.axhline(0, linewidth=1)
plt.ylabel("Contribution across worst sessions (percentage points)")
plt.title("Intraday versus overnight contribution")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 3. Execution robustness

In [ ]:
execution[[
    "total_return", "cagr", "sharpe", "sortino", "max_drawdown", "calmar",
    "turnover_units", "transaction_cost_paid",
    "cagr_delta_vs_baseline", "max_drawdown_delta_vs_baseline",
]]

In [ ]:
plot_exec = execution.loc[[
    "baseline", "all_transitions_delay_1", "risk_increase_delay_1",
    "risk_reduction_delay_1", "baseline_plus_20bps"
]]
plt.figure(figsize=(9, 5))
plt.scatter(plot_exec["max_drawdown"] * 100, plot_exec["cagr"] * 100)
for name, row in plot_exec.iterrows():
    plt.annotate(name, (row["max_drawdown"] * 100, row["cagr"] * 100))
plt.xlabel("Maximum drawdown (%)")
plt.ylabel("CAGR (%)")
plt.title("Execution stress: return versus drawdown")
plt.tight_layout()
plt.show()

## 4. Pre-registered research gate

In [ ]:
gate = summary["research_gate"]
pd.Series(gate["measured"], name="measured").to_frame().join(
    pd.Series(gate["gates"], name="passed")
)

In [ ]:
print("Eligible for continuous state-2 volatility budget:",
      gate["eligible_for_continuous_state2_volatility_budget"])
print("Next direction:", gate["next_direction"])